# Load Model

In [1]:
import os
import numpy as np
import mujoco
from mujoco import mjx
import jax
import myosuite
from myosuite.utils import gym
import jax.numpy as jp
from brax.io import html, mjcf
from stable_baselines3 import PPO
from IPython.display import HTML, display
import matplotlib.pyplot as plt


HERE = os.getcwd()
MODEL_PATH = os.path.join(HERE, "myosuite/envs/myo/assets/arm/myoarm_bionic_bimanual_mjx.xml")
PPO_PATH = os.path.join(HERE, "myosuite/agents/baseline.zip")
print(f"Model path: {MODEL_PATH}")

try:
    mj_model = mujoco.MjModel.from_xml_path(MODEL_PATH)
    print("Model loaded successfully.")
except Exception as e:
    print(f"Failed to load model: {e}")
    # Optionally, you can print the stack trace for more details
    import traceback
    traceback.print_exc()

mj_data   = mujoco.MjData(mj_model)
renderer  = mujoco.Renderer(mj_model)
mjx_model = mjx.put_model(mj_model)
mjx_data  = mjx.put_data(mj_model, mj_data)

MyoSuite:> Registering Myo Envs
Model path: /home/ta747375ki/myosuite/myosuite/envs/myo/assets/arm/myoarm_bionic_bimanual_mjx.xml
Model loaded successfully.


/home/ta747375ki/myosuite/venv-livedemo/lib/python3.10/site-packages/mujoco/mjx/_src/mesh.py:177: UserWarning: Mesh "prosthesis/humerus_lower_top_linkL" has a coplanar face with more than 20 vertices. This may lead to performance issues and inaccuracies in collision detection. Consider decimating the mesh.
  warnings.warn(
/home/ta747375ki/myosuite/venv-livedemo/lib/python3.10/site-packages/mujoco/mjx/_src/mesh.py:177: UserWarning: Mesh "prosthesis/forearm_linkL" has a coplanar face with more than 20 vertices. This may lead to performance issues and inaccuracies in collision detection. Consider decimating the mesh.
  warnings.warn(
/home/ta747375ki/myosuite/venv-livedemo/lib/python3.10/site-packages/mujoco/mjx/_src/mesh.py:177: UserWarning: Mesh "prosthesis/wristy" has a coplanar face with more than 20 vertices. This may lead to performance issues and inaccuracies in collision detection. Consider decimating the mesh.
  warnings.warn(
/home/ta747375ki/myosuite/venv-livedemo/lib/python3.

In [2]:
print('Installing mediapy:')
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
import mediapy as media
import matplotlib.pyplot as plt

scene_option = mujoco.MjvOption()
scene_option.flags[mujoco.mjtVisFlag.mjVIS_JOINT] = True

duration = 3  # (seconds)
framerate = 10  # (Hz)

rollout = []

# MuJoCo Simulation
# mujoco.mj_resetData(mj_model, mj_data)
# while mj_data.time < duration:
#   mujoco.mj_step(mj_model, mj_data)
#   if len(rollout) < mj_data.time * framerate:
#     renderer.update_scene(mj_data, scene_option=scene_option)
#     pixels = renderer.render()
#     rollout.append(pixels)

# MJX Simulation
mjx.reset_data(mjx_model, mjx_data)
while mjx_data.time < duration:
  mjx.step(mjx_model, mjx_data)
  if len(rollout) < mjx_data.time * framerate:
    renderer.update_scene(mjx_data, scene_option=scene_option)
    frame = renderer.render()
    rollout.append(frame)

# Simulate and display video.
media.show_video(rollout, fps=framerate)

Installing mediapy:


/usr/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


AttributeError: module 'mujoco.mjx' has no attribute 'reset_data'

# Define Environment

In [3]:
import jax
from jax import numpy as jp
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from brax.io import html, mjcf, model

class BimanualEnvMJX(PipelineEnv):

  def __init__(
      self,
      forward_reward_weight=1.25,
      ctrl_cost_weight=0.1,
      healthy_reward=5.0,
      terminate_when_unhealthy=True,
      healthy_z_range=(1.0, 2.0),
      reset_noise_scale=1e-2,
      exclude_current_positions_from_observation=True,
      **kwargs,
  ):
    mj_model = mujoco.MjModel.from_xml_path(MODEL_PATH)
    mj_model.opt.solver = mujoco.mjtSolver.mjSOL_CG
    mj_model.opt.iterations = 6
    mj_model.opt.ls_iterations = 6

    sys = mjcf.load_model(mj_model)

    physics_steps_per_control_step = 5
    kwargs['n_frames'] = kwargs.get(
        'n_frames', physics_steps_per_control_step)
    kwargs['backend'] = 'mjx'

    super().__init__(sys, **kwargs)

    self._forward_reward_weight = forward_reward_weight
    self._ctrl_cost_weight = ctrl_cost_weight
    self._healthy_reward = healthy_reward
    self._terminate_when_unhealthy = terminate_when_unhealthy
    self._healthy_z_range = healthy_z_range
    self._reset_noise_scale = reset_noise_scale
    self._exclude_current_positions_from_observation = (
        exclude_current_positions_from_observation
    )

  def reset(self, rng: jp.ndarray) -> State:
    """Resets the environment to an initial state."""
    rng, rng1, rng2 = jax.random.split(rng, 3)

    low, hi = -self._reset_noise_scale, self._reset_noise_scale
    qpos = self.sys.qpos0 + jax.random.uniform(
        rng1, (self.sys.nq,), minval=low, maxval=hi
    )
    qvel = jax.random.uniform(
        rng2, (self.sys.nv,), minval=low, maxval=hi
    )

    data = self.pipeline_init(qpos, qvel)

    obs = self._get_obs(data, jp.zeros(self.sys.nu))
    reward, done, zero = jp.zeros(3)
    metrics = {
        'forward_reward': zero,
        'reward_linvel': zero,
        'reward_quadctrl': zero,
        'reward_alive': zero,
        'x_position': zero,
        'y_position': zero,
        'distance_from_origin': zero,
        'x_velocity': zero,
        'y_velocity': zero,
    }
    return State(data, obs, reward, done, metrics)

  def step(self, state: State, action: jp.ndarray) -> State:
    """Runs one timestep of the environment's dynamics."""
    data0 = state.pipeline_state
    data = self.pipeline_step(data0, action)

    com_before = data0.subtree_com[1]
    com_after = data.subtree_com[1]
    velocity = (com_after - com_before) / self.dt
    forward_reward = self._forward_reward_weight * velocity[0]

    min_z, max_z = self._healthy_z_range
    is_healthy = jp.where(data.q[2] < min_z, 0.0, 1.0)
    is_healthy = jp.where(data.q[2] > max_z, 0.0, is_healthy)
    if self._terminate_when_unhealthy:
      healthy_reward = self._healthy_reward
    else:
      healthy_reward = self._healthy_reward * is_healthy

    ctrl_cost = self._ctrl_cost_weight * jp.sum(jp.square(action))

    obs = self._get_obs(data, action)
    reward = forward_reward + healthy_reward - ctrl_cost
    done = 1.0 - is_healthy if self._terminate_when_unhealthy else 0.0
    state.metrics.update(
        forward_reward=forward_reward,
        reward_linvel=forward_reward,
        reward_quadctrl=-ctrl_cost,
        reward_alive=healthy_reward,
        x_position=com_after[0],
        y_position=com_after[1],
        distance_from_origin=jp.linalg.norm(com_after),
        x_velocity=velocity[0],
        y_velocity=velocity[1],
    )

    return state.replace(
        pipeline_state=data, obs=obs, reward=reward, done=done
    )

  def _get_obs(
      self, data: mjx.Data, action: jp.ndarray
  ) -> jp.ndarray:
    """Observes humanoid body position, velocities, and angles."""
    position = data.qpos
    if self._exclude_current_positions_from_observation:
      position = position[2:]

    # external_contact_forces are excluded
    return jp.concatenate([
        position,
        data.qvel,
        data.cinert[1:].ravel(),
        data.cvel[1:].ravel(),
        data.qfrc_actuator,
    ])


envs.register_environment('myoChallengeBimanualMjx', BimanualEnvMJX)

In [ ]:
env = envs.get_environment('myoChallengeBimanualMjx')
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

# initialize the state
state = jit_reset(jax.random.PRNGKey(0))
rollout = [state.pipeline_state]

# grab a trajectory
for i in range(10):
  ctrl = -0.1 * jp.ones(env.sys.nu)
  state = jit_step(state, ctrl)
  rollout.append(state.pipeline_state)

media.show_video(env.render(rollout, camera='side'), fps=1.0 / env.dt)

In [ ]:
html_content = html.render(mjx_model, rollout, camera="track")
display(HTML(html_content))
with open('output.html', 'w', encoding='utf-8') as file:
    file.write(html_content)

In [1]:
from myosuite.utils import gym
env = gym.make('myoHandPoseRandom-v0')
env.reset()
for _ in range(1000):
    env.mj_render()
    env.step(env.action_space.sample()) # take a random action
env.close()

MyoSuite:> Registering Myo Envs


ModuleNotFoundError: No module named 'termcolor'